# H.4.1 -- maturity distribution table continuous series

Target table: heading text containing `"Maturity Distribution"` (wording
changed across eras -- old: "...of Loans and Securities"; modern: "...of
Securities, Loans, and Selected Other Assets and Liabilities" -- matching
the shorter common phrase is what survives both).

This table turned out to need substantially more structural work than the
balance sheet table before a crosswalk could even be built:
- The raw source HTML for ~2002-2008 releases has a genuine Fed-side bug
  where each `<tr>` redundantly repeats every later row's cells appended
  onto its own, silently producing dozens of garbage columns
  (`Helpers/h41_parse_helpers.py::table_to_grid` now sizes the grid off the
  *mode* of per-row widths instead of the max, which is robust to this).
- The table **flips which axis holds which concept** partway through the
  archive -- maturity buckets as rows / categories as columns through
  2008-10-23, the transpose from 2008-10-30 onward. Normalized here via
  `swap_row_col_before_date`.
- Sub-rows "Holdings" / "Weekly changes" repeat under several different
  parent categories (U.S. Treasury securities, Federal agency securities,
  ...) and are meaningless without re-attaching their parent -- the Fed
  re-encoded which HTML signals that indentation at least four different
  ways across the archive, so this is now special-cased by label rather
  than chased era by era (see `grid_to_tidy`'s `enable_parent_prefix`,
  which defaults OFF since the balance sheet table's labels don't need it
  and its already-saved crosswalk depends on the un-prefixed form).
- Several eras use different "zero/NA" placeholder conventions
  (`-`/`--` vs `---` vs `...`), which previously dragged some data rows'
  numeric fraction below the header-detection threshold and got them
  wrongly absorbed into the column headers.

Net effect: parse_failed rate 13.8% -> 0.40%, row labels 77 -> 75 (now
clean asset categories throughout), column labels 274 -> 23 (now clean
maturity buckets throughout).

Known residual limitation: a handful of long crisis-facility names (Main
Street Lending Program, Municipal Liquidity Facility, CPFF) wrap across
2-3 physical HTML rows in a way the continuation-merge logic can't fully
reconstruct -- their crosswalk entries map from truncated label fragments,
confirmed by date range rather than exact wording. These are all small,
short-lived facilities; not worth further parser engineering.

In [1]:
import os, sys, glob
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), "Helpers"))
from h41_parse_helpers import (
    extract_tidy_many, add_clean_value, label_inventory, unmapped_labels,
    load_or_init_crosswalk, apply_crosswalk, check_components_sum_to_total,
    coverage_report, swap_row_col_before_date, strip_footnote_marker, save_wide_excel,
)
import pandas as pd

BASE = os.path.dirname(os.getcwd())
HTML_DIR = os.path.join(BASE, "Data Input", "Raw Input", "h41_html")
CROSSWALK_DIR = os.path.join(BASE, "Data Input", "Raw Input", "crosswalks")
os.makedirs(CROSSWALK_DIR, exist_ok=True)
ROW_CROSSWALK_PATH = os.path.join(CROSSWALK_DIR, "maturity_row_crosswalk.xlsx")  # asset categories
COL_CROSSWALK_PATH = os.path.join(CROSSWALK_DIR, "maturity_col_crosswalk.xlsx")  # maturity buckets

# NOTE: the Fed re-worded this heading at some point (old: "Maturity
# Distribution of Loans and Securities"; modern: "Maturity Distribution of
# Securities, Loans, and Selected Other Assets and Liabilities"). Matching
# just "Maturity Distribution" is what makes this work across both eras.
SUMMARY_TARGET = "Maturity Distribution"

# This table flips which axis holds which concept partway through the
# archive: releases through 2008-10-23 report maturity buckets as rows and
# asset categories as columns; releases from 2008-10-30 onward report the
# transpose. swap_row_col_before_date() normalizes everything to the
# modern (majority) convention: row = asset category, col = maturity bucket.
ORIENTATION_FLIP_DATE = pd.Timestamp("2008-10-30")

all_files = sorted(glob.glob(os.path.join(HTML_DIR, "h41_*.html")))
print(f"{len(all_files)} release files found")

1544 release files found


## Step 1 -- extract every release to long/tidy form

In [2]:
tidy = extract_tidy_many(all_files, SUMMARY_TARGET, enable_parent_prefix=True)
tidy = swap_row_col_before_date(tidy, ORIENTATION_FLIP_DATE)
# Parent-prefixing happens before the swap, keyed off whichever axis was
# originally the row axis; footnote markers on what were originally COLUMN
# labels (pre-2008 asset categories) never got stripped during extraction
# as a result. Re-apply uniformly now that both eras share one axis.
tidy["row_label_raw"] = tidy["row_label_raw"].map(strip_footnote_marker)
tidy = add_clean_value(tidy)
print(f"{len(tidy):,} tidy rows across {tidy['date'].nunique()} releases")
print(f"parse_failed rate: {tidy['parse_failed'].mean():.2%}")
tidy.head(10)

133,714 tidy rows across 1544 releases
parse_failed rate: 0.40%


,date,row_label_raw,col_label_raw,value_raw,source,value,parse_failed
0,1997-03-05,Loans,Within 15 day,10,h41_19970306.html,10.0,False
1,1997-03-05,Acceptances,Within 15 day,0,h41_19970306.html,0.0,False
2,1997-03-05,col_2,Within 15 day,Within 15 days,h41_19970306.html,NaN,True
3,1997-03-05,U.S. government securities (1) - Holdings,Within 15 day,"19,290",h41_19970306.html,19290.0,False
4,1997-03-05,U.S. government securities (1) - Weekly changes,Within 15 day,"- 9,138",h41_19970306.html,-9138.0,False
5,1997-03-05,Federal agency obligations - Holdings,Within 15 day,635,h41_19970306.html,635.0,False
6,1997-03-05,Federal agency obligations - Weekly changes,Within 15 day,- 2013,h41_19970306.html,-2013.0,False
7,1997-03-05,Loans,16 days to,19,h41_19970306.html,19.0,False
8,1997-03-05,Acceptances,16 days to,0,h41_19970306.html,0.0,False
9,1997-03-05,col_2,16 days to,16 days to 90 days,h41_19970306.html,NaN,True


## Step 2 -- label inventory and crosswalk gap detection

Row labels are now asset categories (Loans, U.S. Treasury securities,
Federal agency debt securities, Mortgage-backed securities, Repurchase
agreements, crisis-facility names, ... -- including the Holdings/Weekly
changes split re-attached to its parent category). Column labels are
maturity buckets (Within 15 days, 16 days to 90 days, ..., Over 10 years,
plus a Total/All aggregate column).

In [3]:
row_inventory = label_inventory(tidy, "row_label_raw")
col_inventory = label_inventory(tidy, "col_label_raw")

row_crosswalk = load_or_init_crosswalk(ROW_CROSSWALK_PATH)
col_crosswalk = load_or_init_crosswalk(COL_CROSSWALK_PATH)

row_unmapped = unmapped_labels(row_inventory, row_crosswalk, "row_label_raw")
col_unmapped = unmapped_labels(col_inventory, col_crosswalk, "col_label_raw")

print(f"{len(row_unmapped)} / {len(row_inventory)} row (asset category) labels unmapped")
print(f"{len(col_unmapped)} / {len(col_inventory)} column (maturity bucket) labels unmapped")

row_unmapped.rename(columns={"row_label_raw": "raw"}).assign(standardized="").to_excel(
    os.path.join(CROSSWALK_DIR, "maturity_row_UNMAPPED_review.xlsx"), index=False)
col_unmapped.rename(columns={"col_label_raw": "raw"}).assign(standardized="").to_excel(
    os.path.join(CROSSWALK_DIR, "maturity_col_UNMAPPED_review.xlsx"), index=False)
row_unmapped

7 / 82 row (asset category) labels unmapped
12 / 35 column (maturity bucket) labels unmapped


,row_label_raw,first_seen,last_seen,n_obs
0,Within 15 days,2008-10-29,2008-10-29,12
1,Over 5 years to 10 years,2008-10-29,2008-10-29,12
2,Over 10 years,2008-10-29,2008-10-29,12
3,Over 1 year to 5 years,2008-10-29,2008-10-29,12
4,16 days to 90 days,2008-10-29,2008-10-29,12
5,91 days to 1 year,2008-10-29,2008-10-29,12
6,All,2008-10-29,2008-10-29,12


## Step 3 -- apply crosswalk (once populated)

In [4]:
tidy_mapped = apply_crosswalk(tidy, row_crosswalk, "row_label_raw", out_col="asset_category")
tidy_mapped = apply_crosswalk(tidy_mapped, col_crosswalk, "col_label_raw", out_col="maturity_bucket")

fully_mapped = tidy_mapped.dropna(subset=["asset_category", "maturity_bucket"])
print(f"{len(fully_mapped):,} / {len(tidy_mapped):,} rows fully mapped on both axes "
      f"({len(fully_mapped)/len(tidy_mapped):.2%})")

133,070 / 133,714 rows fully mapped on both axes (99.52%)


## Step 4 -- pivot to wide, once the crosswalk covers enough of the data

In [5]:
wide = (
    fully_mapped
    .groupby(["date", "asset_category", "maturity_bucket"])["value"].sum()
    .unstack("maturity_bucket")
    .sort_index()
)
print(f"wide shape: {wide.shape}, {wide.index.get_level_values('asset_category').nunique()} asset categories")
wide.head()

wide shape: (15481, 8), 23 asset categories


maturity_bucket                                     MATURITY_16_TO_90_DAYS  \
date       asset_category                                                    
1997-03-05 ACCEPTANCES                                                 0.0   
           FEDERAL_AGENCY_SECURITIES_HOLDINGS                        762.0   
           FEDERAL_AGENCY_SECURITIES_WEEKLY_CHANGE                   307.0   
           LOANS                                                      19.0   
           UST_SECURITIES_HOLDINGS                                 95624.0   

maturity_bucket                                     MATURITY_1_TO_5_YEARS  \
date       asset_category                                                   
1997-03-05 ACCEPTANCES                                                NaN   
           FEDERAL_AGENCY_SECURITIES_HOLDINGS                         NaN   
           FEDERAL_AGENCY_SECURITIES_WEEKLY_CHANGE                    NaN   
           LOANS                                                      NaN   
           UST_SECURITIES_HOLDINGS                                    NaN   

maturity_bucket                                     MATURITY_5_TO_10_YEARS  \
date       asset_category                                                    
1997-03-05 ACCEPTANCES                                                 NaN   
           FEDERAL_AGENCY_SECURITIES_HOLDINGS                          NaN   
           FEDERAL_AGENCY_SECURITIES_WEEKLY_CHANGE                     NaN   
           LOANS                                                       NaN   
           UST_SECURITIES_HOLDINGS                                     NaN   

maturity_bucket                                     MATURITY_91_DAYS_TO_1_YEAR  \
date       asset_category                                                        
1997-03-05 ACCEPTANCES                                                     NaN   
           FEDERAL_AGENCY_SECURITIES_HOLDINGS                              NaN   
           FEDERAL_AGENCY_SECURITIES_WEEKLY_CHANGE                         NaN   
           LOANS                                                           NaN   
           UST_SECURITIES_HOLDINGS                                         NaN   

maturity_bucket                                     MATURITY_OVER_10_YEARS  \
date       asset_category                                                    
1997-03-05 ACCEPTANCES                                                 NaN   
           FEDERAL_AGENCY_SECURITIES_HOLDINGS                          NaN   
           FEDERAL_AGENCY_SECURITIES_WEEKLY_CHANGE                     NaN   
           LOANS                                                       NaN   
           UST_SECURITIES_HOLDINGS                                     NaN   

maturity_bucket                                     MATURITY_OVER_90_DAYS  \
date       asset_category                                                   
1997-03-05 ACCEPTANCES                                                0.0   
           FEDERAL_AGENCY_SECURITIES_HOLDINGS                      2904.0   
           FEDERAL_AGENCY_SECURITIES_WEEKLY_CHANGE                -1680.0   
           LOANS                                                     28.0   
           UST_SECURITIES_HOLDINGS                               517181.0   

maturity_bucket                                     MATURITY_TOTAL  \
date       asset_category                                            
1997-03-05 ACCEPTANCES                                         NaN   
           FEDERAL_AGENCY_SECURITIES_HOLDINGS                  NaN   
           FEDERAL_AGENCY_SECURITIES_WEEKLY_CHANGE             NaN   
           LOANS                                               NaN   
           UST_SECURITIES_HOLDINGS                             NaN   

maturity_bucket                                     MATURITY_WITHIN_15_DAYS  
date       asset_category                                                    
1997-03-05 ACCEPTANCES                                   

## Step 5 -- reconciliation: the table's own "Total"/"All" bucket

The maturity table reports a Total/All bucket per asset category that
should equal the sum of the individual maturity buckets -- a built-in
check the old notebook didn't exploit (it caught bad weeks by eye, by
accident, and only found 2). A 5 ($ million) tolerance absorbs ordinary
rounding in the Fed's own published figures.

In [6]:
MATURITY_BUCKETS = [
    "MATURITY_WITHIN_15_DAYS", "MATURITY_16_TO_90_DAYS", "MATURITY_91_DAYS_TO_1_YEAR",
    "MATURITY_1_TO_5_YEARS", "MATURITY_5_TO_10_YEARS", "MATURITY_OVER_10_YEARS",
]

recon_results = {}
for cat in wide.index.get_level_values("asset_category").unique():
    sub = wide.xs(cat, level="asset_category")
    have_buckets = [b for b in MATURITY_BUCKETS if b in sub.columns]
    if "MATURITY_TOTAL" not in sub.columns or len(have_buckets) < 3:
        continue  # not enough of the table's own bucket breakdown to check against
    bad_weeks = check_components_sum_to_total(sub, component_cols=have_buckets, total_col="MATURITY_TOTAL", tolerance=5.0)
    recon_results[cat] = (len(sub), len(bad_weeks))

recon_df = pd.DataFrame(
    [(cat, n, bad, bad / n) for cat, (n, bad) in recon_results.items()],
    columns=["asset_category", "n_weeks", "bad_weeks", "pct_bad"],
).sort_values("pct_bad", ascending=False)
print(recon_df.to_string(index=False))

                              asset_category  n_weeks  bad_weeks  pct_bad
     FEDERAL_AGENCY_SECURITIES_WEEKLY_CHANGE      577          7 0.012132
                                 ACCEPTANCES      133          0 0.000000
          FEDERAL_AGENCY_SECURITIES_HOLDINGS      577          0 0.000000
                                       LOANS     1368          0 0.000000
                     UST_SECURITIES_HOLDINGS     1541          0 0.000000
                UST_SECURITIES_WEEKLY_CHANGE     1541          0 0.000000
                             REPO_ASSET_SIDE     1408          0 0.000000
                 REVERSE_REPO_LIABILITY_SIDE     1408          0 0.000000
                                 OTHER_LOANS      173          0 0.000000
                     TAF_TERM_AUCTION_CREDIT      124          0 0.000000
                           CPFF_LLC_HOLDINGS       95          0 0.000000
                          MMIFF_LLC_HOLDINGS       49          0 0.000000
     FEDERAL_AGENCY_DEBT_SECURITIES_HO

In [7]:
coverage = (
    fully_mapped.groupby("asset_category")["date"]
    .agg(first_seen="min", last_seen="max", n_weeks="nunique")
    .sort_values("first_seen")
)
coverage

,first_seen,last_seen,n_weeks
asset_category,,,
ACCEPTANCES,1997-03-05,1999-09-29,133
UST_SECURITIES_HOLDINGS,1997-03-05,2026-09-30,1541
LOANS,1997-03-05,2026-09-30,1368
FEDERAL_AGENCY_SECURITIES_WEEKLY_CHANGE,1997-03-05,2009-01-07,577
FEDERAL_AGENCY_SECURITIES_HOLDINGS,1997-03-05,2009-01-07,577
UST_SECURITIES_WEEKLY_CHANGE,1997-03-05,2026-09-30,1541
REVERSE_REPO_LIABILITY_SIDE,1999-10-06,2026-09-30,1408
REPO_ASSET_SIDE,1999-10-06,2026-09-30,1408
OTHER_LOANS,2007-12-26,2011-04-20,173


## Step 6 -- save to Data Output

Tidy format, matching the balance sheet deliverable's shape: `date`,
`asset_category`, `maturity_bucket`, `value`. `MATURITY_TOTAL` rows are
kept in (the table's own reported total, used above for reconciliation,
not re-derived) rather than dropped.

In [8]:
out_cols = ["date", "asset_category", "maturity_bucket", "value"]
out = fully_mapped[out_cols].sort_values(["asset_category", "maturity_bucket", "date"]).reset_index(drop=True)

out_path = os.path.join(BASE, "Data Output", "H41_maturity_distribution_clean_timeseries.xlsx")
out.to_excel(out_path, index=False)
print(f"Saved {len(out):,} rows to: {out_path}")

Saved 133,070 rows to: C:\Users\User\Desktop\DISSERTATION PAPERS\PAPER 1 - STRUCTURAL BREAKS\Data Output\H41_maturity_distribution_clean_timeseries.xlsx


## Step 7 -- pivot to the paper's wide deliverable shape

Dates in columns, `(asset_category, maturity_bucket)` in rows -- matching JHU's own spreadsheet structure. `MATURITY_TOTAL` is kept in as its own row (the table's reported total).

In [9]:
wide_out_path = os.path.join(BASE, "Data Output", "H41_maturity_distribution_1997_2026_levels_wide.xlsx")
wide_final = save_wide_excel(fully_mapped, index_cols=["asset_category", "maturity_bucket"], out_path=wide_out_path)

Saved 162 rows x 1,541 date columns to: C:\Users\User\Desktop\DISSERTATION PAPERS\PAPER 1 - STRUCTURAL BREAKS\Data Output\H41_maturity_distribution_1997_2026_levels_wide.xlsx
